# ResearchAgent v3 — State-Based Multi-Agent Pipeline
Multi-agent research pipeline with direct `ResearchState` object handoffs.  
**No temporary files** between agents — all inter-agent data flows through a single typed state object.

In [ ]:
!pip install torch transformers bs4 pymupdf langchain-text-splitters chromadb \
    sentence-transformers networkx matplotlib semanticscholar huggingface_hub \
    datasets scikit-learn tqdm bitsandbytes>=0.46.1
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
import os, re, sys, json, time, ast, gc, shutil, subprocess, gzip, sqlite3
import traceback as _traceback
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import requests
import networkx as nx
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import HfApi
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions


class SharedModelManager:
    def __init__(self, model_id="google/gemma-4-12B-it"):
        self.model_id = model_id
        self.model = None
        self.processor = None

    def get_model(self):
        """Loads or returns the active Gemma instance in 4-bit quantization."""
        if self.model is None or self.processor is None:
            print(f"🧠 Loading {self.model_id} (first-time initialisation)...")
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
            )
            self.processor = AutoProcessor.from_pretrained(self.model_id)
            self.model = AutoModelForMultimodalLM.from_pretrained(
                self.model_id, device_map="auto", quantization_config=quant_config
            )
            print("✅ Model loaded.")
        else:
            print("⚡ Reusing active VRAM model reference.")
        return self.model, self.processor

    def unload(self):
        """Purges Gemma from memory to release VRAM."""
        print("🗑️ Purging Gemma from memory...")
        if self.model is not None:
            del self.model; self.model = None
        if self.processor is not None:
            del self.processor; self.processor = None
        gc.collect()
        torch.cuda.empty_cache()
        print("✅ GPU memory flushed.")


llm_manager = SharedModelManager()

## ResearchState — Single Shared Data Object
All agents read from and write to this object.  No files cross agent boundaries.

In [ ]:
@dataclass
class ResearchState:
    """
    Centralised state container threaded through every pipeline agent.

    Fields
    ------
    topic      : The user's research question / primary arXiv query.
    literature : Output of the Literature Agent — papers, KG summary, corpus stats.
    hypothesis : Output of the Hypothesis Agent — ranked hypotheses + best selection.
    experiment : Output of the Experiment Agent — generated code, run log, metrics.
    analysis   : Output of the Analysis Agent  — verdict, interpretation, next step.
    paper      : Final LaTeX paper produced by the Writing Agent (revised by Critic).
    """
    topic      : str               = ''
    literature : Dict[str, Any]    = field(default_factory=dict)
    hypothesis : Dict[str, Any]    = field(default_factory=dict)
    experiment : Dict[str, Any]    = field(default_factory=dict)
    analysis   : Dict[str, Any]    = field(default_factory=dict)
    paper      : str               = ''

    def summary(self):
        """Quick human-readable status overview."""
        lines = [
            f"topic      : {self.topic[:80] or '(empty)'}",
            f"literature : {len(self.literature.get('papers', []))} papers, "
            f"{self.literature.get('chroma_count', 0)} chunks",
            f"hypothesis : {len(self.hypothesis.get('hypotheses', []))} generated, "
            f"best='{self.hypothesis.get('best', {}).get('title', 'none')}'",
            f"experiment : status={self.experiment.get('final_status', 'none')}, "
            f"metrics={self.experiment.get('metrics')}",
            f"analysis   : verdict={self.analysis.get('verdict', 'none')}",
            f"paper      : {len(self.paper)} chars",
        ]
        print("\n".join(lines))

## Shared Infrastructure
ChromaDB vector store, knowledge graph, and Semantic Scholar helpers shared by all agents.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
S2_API_KEY        = os.environ.get("S2_API_KEY", "")
S2_HEADERS        = {"x-api-key": S2_API_KEY} if S2_API_KEY else {}
S2_GRAPH_URL      = "https://api.semanticscholar.org/graph/v1"
S2_REC_URL        = "https://api.semanticscholar.org/recommendations/v1"
S2_DATA_URL       = "https://api.semanticscholar.org/datasets/v1"
DATASET_CACHE_DIR = "./s2_dataset_cache"
DOWNLOAD_DIR      = "arxiv_papers"
DB_DIR            = "./lit_review_db"
MAX_RESULTS       = 15
SANDBOX_DIR       = "experiment_sandbox"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"]       = "expandable_segments:True"

# ── ChromaDB ─────────────────────────────────────────────────────────────────
print("Initialising ChromaDB vector store...")
chroma_client = chromadb.PersistentClient(path=DB_DIR)
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-small-en-v1.5"
)
collection = chroma_client.get_or_create_collection(
    name="arxiv_papers", embedding_function=sentence_transformer_ef
)

# ── Knowledge Graph ───────────────────────────────────────────────────────────
kg = nx.Graph()

# ── Relevance model (lazy-loaded) ─────────────────────────────────────────────
_relevance_model = None

def _get_relevance_model():
    global _relevance_model
    if _relevance_model is None:
        _relevance_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    return _relevance_model


# ── S2 HTTP helper ────────────────────────────────────────────────────────────
def _s2_get(path, base=S2_GRAPH_URL, params=None, retries=3):
    url = base + path
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=S2_HEADERS, params=params, timeout=20)
            if r.status_code == 429:
                wait = 10 * attempt
                print(f"   ⏳ Rate-limited — waiting {wait}s (attempt {attempt}/{retries})")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            if attempt == retries:
                print(f"   ⚠️  S2 GET failed: {e}")
                return None
            time.sleep(3 * attempt)
    return None


# ── Relevance filter ──────────────────────────────────────────────────────────
def _filter_by_relevance(papers, research_question, threshold=0.35):
    model   = _get_relevance_model()
    q_emb   = model.encode(research_question, convert_to_tensor=True)
    scored  = []
    for p in papers:
        text  = p.get("abstract") or p.get("title") or ""
        score = float(util.cos_sim(q_emb, model.encode(text, convert_to_tensor=True)))
        scored.append((score, p))
    kept = [(s, p) for s, p in scored if s >= threshold]
    kept.sort(key=lambda x: x[0], reverse=True)
    for score, p in kept:
        p["relevance_score"] = round(score, 3)
    return [p for _, p in kept]


print("✅ Shared infrastructure ready.")

## Semantic Scholar API Helpers

In [ ]:
def _resolve_s2_ids(papers):
    """Batch-convert arXiv IDs to Semantic Scholar Paper IDs.

    arXiv search returns versioned IDs (e.g. "2301.12345v1") but the S2
    batch endpoint only recognises the canonical, version-less form
    ("2301.12345").  We therefore:
      1. Strip version suffixes before sending to S2.
      2. Re-map S2's canonical ArXiv IDs back to the original versioned
         keys so all downstream callers can look up by p["id"] without
         any extra stripping.
    """
    # raw IDs as stored on each paper dict (may carry "ss_" prefix)
    raw_ids = [p["id"].replace("ss_", "").replace("rec_", "")
               for p in papers if p.get("id")]
    if not raw_ids:
        return {}

    # versioned_id -> canonical (version-stripped) id
    versioned_to_canonical = {
        vid: re.sub(r"v\d+$", "", vid) for vid in raw_ids
    }
    canonical_ids = list(set(versioned_to_canonical.values()))

    canonical_map = {}   # canonical_arxiv_id -> S2 paperId
    for chunk_start in range(0, len(canonical_ids), 500):
        chunk   = canonical_ids[chunk_start:chunk_start + 500]
        payload = {"ids": [f"ARXIV:{cid}" for cid in chunk]}
        try:
            r = requests.post(
                f"{S2_GRAPH_URL}/paper/batch",
                headers={**S2_HEADERS, "Content-Type": "application/json"},
                params={"fields": "paperId,externalIds"},
                json=payload, timeout=30
            )
            if r.status_code == 200:
                for item in r.json():
                    if item and item.get("paperId"):
                        ext      = item.get("externalIds") or {}
                        arxiv_id = ext.get("ArXiv", "")
                        if arxiv_id:
                            canonical_map[arxiv_id] = item["paperId"]
        except requests.RequestException as e:
            print(f"   ⚠️  Batch ID resolution failed: {e}")
        time.sleep(0.5)

    # Rebuild id_map keyed by the ORIGINAL versioned IDs so callers need
    # no changes – a versioned key not in S2 is simply omitted.
    id_map = {
        vid: canonical_map[cid]
        for vid, cid in versioned_to_canonical.items()
        if cid in canonical_map
    }
    print(f"🔗 Resolved {len(id_map)}/{len(raw_ids)} arXiv IDs to S2 IDs")
    return id_map


def get_recommendations(positive_papers, negative_papers=None, limit=50,
                         research_question=""):
    """Call the S2 Recommendations API using seed papers."""
    print(f"\n🎯 Fetching recommendations from {len(positive_papers)} seeds...")
    all_seeds  = (positive_papers or []) + (negative_papers or [])
    id_map     = _resolve_s2_ids(all_seeds)
    pos_s2_ids = [id_map[p["id"].replace("ss_","").replace("rec_","")]
                  for p in positive_papers
                  if p.get("id","").replace("ss_","").replace("rec_","") in id_map]
    neg_s2_ids = [id_map[p["id"].replace("ss_","").replace("rec_","")]
                  for p in (negative_papers or [])
                  if p.get("id","").replace("ss_","").replace("rec_","") in id_map]
    if not pos_s2_ids:
        print("   ⚠️  No S2 IDs resolved — skipping recommendations.")
        return []
    payload = {"positivePaperIds": pos_s2_ids[:50], "negativePaperIds": neg_s2_ids[:20]}
    params  = {"fields": "title,abstract,openAccessPdf,externalIds,citationCount,year",
               "limit": min(limit, 500)}
    try:
        r = requests.post(f"{S2_REC_URL}/papers",
                          headers={**S2_HEADERS, "Content-Type": "application/json"},
                          params=params, json=payload, timeout=30)
        if r.status_code == 429:
            print("   ⏳ Rate-limited — waiting 15s...")
            time.sleep(15)
            r = requests.post(f"{S2_REC_URL}/papers",
                              headers={**S2_HEADERS, "Content-Type": "application/json"},
                              params=params, json=payload, timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"   ⚠️  Recommendations API error: {e}")
        return []
    raw_recs = r.json().get("recommendedPapers", [])
    print(f"   📬 {len(raw_recs)} raw recommendations received")
    papers = []
    for rec in raw_recs:
        pdf_url  = (rec.get("openAccessPdf") or {}).get("url")
        if not pdf_url:
            continue
        ext      = rec.get("externalIds") or {}
        arxiv_id = ext.get("ArXiv") or rec["paperId"]
        papers.append({
            "title": rec.get("title", ""), "abstract": rec.get("abstract") or "",
            "pdf_url": pdf_url, "id": f"rec_{arxiv_id}",
            "citations": rec.get("citationCount") or 0,
            "year": rec.get("year"), "source": "s2_recommendations",
        })
    papers.sort(key=lambda p: p["citations"], reverse=True)
    if research_question and papers:
        papers = _filter_by_relevance(papers, research_question, threshold=0.30)
    print(f"✅ Recommendations: {len(papers)} open-access papers after filtering")
    return papers

In [ ]:
def _bulk_insert(cur, dataset_name, batch):
    if dataset_name == "abstracts":
        cur.executemany(
            "INSERT OR REPLACE INTO abstracts (corpusid, arxiv_id, abstract) VALUES (?,?,?)", batch)
    elif dataset_name == "tldrs":
        cur.executemany(
            "INSERT OR REPLACE INTO tldrs (corpusid, tldr_text) VALUES (?,?)", batch)
    elif dataset_name == "papers":
        cur.executemany(
            "INSERT OR REPLACE INTO papers (corpusid, arxiv_id, title, year, citations) VALUES (?,?,?,?,?)", batch)
    else:
        cur.executemany(
            "INSERT OR REPLACE INTO records (corpusid, json_data) VALUES (?,?)", batch)


class S2DatasetManager:
    """Manages local downloads of Semantic Scholar bulk datasets for offline abstract lookup."""

    def __init__(self, cache_dir=DATASET_CACHE_DIR, api_key=S2_API_KEY):
        self.cache_dir = cache_dir
        self.headers   = {"x-api-key": api_key} if api_key else {}
        os.makedirs(cache_dir, exist_ok=True)
        self._db_conn  = {}

    def get_abstracts_for_papers(self, papers):
        """Batch-lookup abstracts from the local 'abstracts' index; falls back to API."""
        conn      = self._get_conn("abstracts")
        cur       = conn.cursor()
        paper_map = {p["id"].replace("ss_","").replace("rec_",""): p for p in papers}
        ids_str   = ",".join(f"'{aid}'" for aid in paper_map)
        rows      = cur.execute(
            f"SELECT arxiv_id, abstract FROM abstracts WHERE arxiv_id IN ({ids_str})"
        ).fetchall()
        found = set()
        for arxiv_id, abstract in rows:
            if arxiv_id in paper_map and abstract:
                paper_map[arxiv_id]["abstract"] = abstract
                found.add(arxiv_id)
        missing = [aid for aid in paper_map if aid not in found]
        if missing:
            print(f"   🔍 {len(missing)} abstracts not in local index — fetching from API...")
            self._fetch_abstracts_from_api(missing, paper_map)
        print(f"📖 Abstracts: {len(found)} from index, {len(missing)} from API")
        return list(paper_map.values())

    def _fetch_abstracts_from_api(self, arxiv_ids, paper_map):
        for arxiv_id in arxiv_ids:
            data = _s2_get(f"/paper/ARXIV:{arxiv_id}", params={"fields": "abstract"})
            if data and data.get("abstract"):
                paper_map[arxiv_id]["abstract"] = data["abstract"]
            time.sleep(1.1)

    def _get_conn(self, dataset_name):
        if dataset_name not in self._db_conn:
            db_path = os.path.join(self.cache_dir, f"{dataset_name}.db")
            if not os.path.exists(db_path):
                raise FileNotFoundError(
                    f"No index for '{dataset_name}'. Run build_index('{dataset_name}') first.")
            self._db_conn[dataset_name] = sqlite3.connect(db_path)
        return self._db_conn[dataset_name]

    # download_dataset / build_index / apply_incremental_diff omitted for brevity
    # (unchanged from v2 — see original notebook for full implementation)


dataset_manager = S2DatasetManager()
print("✅ S2DatasetManager ready.")

## Agent 1: Literature Agent

In [ ]:
def _ask_gemma_for_queries(research_question, n=4):
    """Generate n diverse arXiv search queries from the research question."""
    model, processor = llm_manager.get_model()
    messages = [
        {"role": "system", "content": "You are a research librarian."},
        {"role": "user", "content": (
            f"Research question: '{research_question}'\n"
            f"Output EXACTLY {n} diverse arXiv search queries, one per line. "
            "Vary angles: core terms, synonyms, related methods, subtopics. "
            "No numbering, no explanation."
        )},
    ]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150)
    raw = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    queries = [q.strip() for q in raw.strip().splitlines() if q.strip()][:n]
    print(f"🧠 Gemma generated {len(queries)} queries: {queries}")
    return queries


def _search_arxiv_with_abstract(query, max_results=12):
    import urllib.parse
    formatted = urllib.parse.quote(f"all:{query}")
    url = (f"http://export.arxiv.org/api/query"
           f"?search_query={formatted}&start=0&max_results={max_results}")
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"   ⚠️  arXiv search failed: {e}")
        return []
    soup   = BeautifulSoup(r.text, "xml")
    papers = []
    for entry in soup.find_all("entry"):
        title    = entry.title.text.strip().replace("\n", " ")
        pdf_url  = next(
            (link.get("href") for link in entry.find_all("link")
             if link.get("title") == "pdf"), None
        )
        paper_id = entry.id.text.split("/")[-1]
        abstract = entry.find("summary")
        abstract = abstract.text.strip() if abstract else ""
        if pdf_url:
            papers.append({"title": title, "abstract": abstract,
                           "pdf_url": pdf_url, "id": paper_id, "source": "arxiv"})
    return papers


def _download_paper_with_retry(paper, download_dir, retries=3):
    safe_id  = paper["id"].replace("/", "_").replace(":", "_")
    filepath = os.path.join(download_dir, f"{safe_id}.pdf")
    if os.path.exists(filepath) and os.path.getsize(filepath) > 5000:
        return filepath
    os.makedirs(download_dir, exist_ok=True)
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(paper["pdf_url"], stream=True, timeout=60)
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            if os.path.getsize(filepath) < 5000:
                raise ValueError("File too small — likely not a valid PDF")
            return filepath
        except Exception as e:
            print(f"   ⚠️  Attempt {attempt}/{retries}: {e}")
            if attempt < retries:
                time.sleep(2 ** attempt)
    return None


def _download_papers_parallel(papers, download_dir, workers=4):
    print(f"\n⬇️  Downloading {len(papers)} papers...")
    results = {}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_download_paper_with_retry, p, download_dir): p for p in papers}
        for future in as_completed(futures):
            paper  = futures[future]
            path   = future.result()
            status = "✅" if path else "❌"
            print(f"   {status} {paper['title'][:60]}")
            if path:
                results[paper["id"]] = path
    print(f"📥 Downloaded {len(results)}/{len(papers)} papers")
    return results


def extract_and_chunk_pdf(filepath):
    doc       = fitz.open(filepath)
    full_text = "".join(page.get_text() for page in doc)
    splitter  = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    return splitter.split_text(full_text)


def store_chunks_in_db(chunks, paper_title, paper_id):
    if not chunks:
        return
    metadatas = [{"title": paper_title, "chunk_index": i} for i in range(len(chunks))]
    ids       = [f"{paper_id}_chunk_{i}" for i in range(len(chunks))]
    collection.add(documents=chunks, metadatas=metadatas, ids=ids)
    print(f"   💾 Stored {len(chunks)} chunks.")


def extract_paper_insights(paper_title):
    results = collection.query(
        query_texts=[f"key findings, methodology, and research gaps in {paper_title}"],
        n_results=5, where={"title": paper_title}
    )
    if not results["documents"] or not results["documents"][0]:
        return "Unknown Method", "Unknown Gap"
    context = "\n".join(results["documents"][0])
    model, processor = llm_manager.get_model()
    messages = [
        {"role": "system", "content":
         "Extract METHOD and GAP. Format:\nMETHOD: <desc>\nGAP: <desc>"},
        {"role": "user", "content": f"Text:\n{context}"},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100)
    insights = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    method, gap = "Unspecified Method", "Unspecified Gap"
    for line in insights.split("\n"):
        if line.startswith("METHOD:"): method = line.replace("METHOD:", "").strip()
        if line.startswith("GAP:"):    gap    = line.replace("GAP:",    "").strip()
    return method, gap


def update_knowledge_graph(paper_title, methodology, gap):
    short = paper_title[:30] + "..."
    kg.add_node(short, type="Paper")
    if methodology and methodology != "Unspecified Method":
        kg.add_node(methodology, type="Method")
        kg.add_edge(short, methodology, relation="USES")
    if gap and gap != "Unspecified Gap":
        kg.add_node(gap, type="Gap")
        kg.add_edge(short, gap, relation="ADDRESSES")
    print(f"   🕸️ Added '{short}' to KG.")


def visualize_graph():
    print("🎨 Generating Knowledge Graph...")
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(kg, seed=42)
    color_map = []
    for _, attr in kg.nodes(data=True):
        t = attr.get("type")
        color_map.append("skyblue" if t=="Paper" else "lightgreen" if t=="Method" else "salmon")
    nx.draw(kg, pos, with_labels=True, node_color=color_map,
            node_size=3000, font_size=8, font_weight="bold", edge_color="gray")
    plt.title("Literature Review Knowledge Graph")
    plt.savefig("knowledge_graph.png")
    print("✅ Graph saved as 'knowledge_graph.png'.")

In [ ]:
def run_literature_agent(state: ResearchState) -> ResearchState:
    """
    Agent 1 — Literature Review.

    Reads  : state.topic  (prompts user if empty)
    Writes : state.topic, state.literature
    """
    print("\n=== Agent 1: Literature Review Agent ===")

    if not state.topic:
        state.topic = input("Enter your research question: ")

    # ── Step 1: generate diverse queries ────────────────────────────────────
    queries = _ask_gemma_for_queries(state.topic, n=4)

    # ── Step 2: multi-query arXiv search ────────────────────────────────────
    papers, seen_ids = [], set()
    for query in queries:
        for p in _search_arxiv_with_abstract(query, max_results=12):
            if p["id"] not in seen_ids:
                seen_ids.add(p["id"])
                papers.append(p)
    print(f"📄 arXiv: {len(papers)} unique candidate papers")

    # ── Step 3: enrich abstracts via Datasets API (if local index available) ─
    abstracts_db = os.path.join(DATASET_CACHE_DIR, "abstracts.db")
    if os.path.exists(abstracts_db):
        papers = dataset_manager.get_abstracts_for_papers(papers)
    else:
        print("💡 Tip: build a local abstracts index for faster lookups.")

    # ── Step 4: relevance filter before downloading ──────────────────────────
    papers = _filter_by_relevance(papers, state.topic, threshold=0.32)
    print(f"🎯 After relevance filter: {len(papers)} papers kept")

    # ── Step 5: Recommendations API — seed with top papers ──────────────────
    top_seeds      = papers[:10]
    negative_seeds = [p for p in papers if p.get("relevance_score", 1) < 0.25][:5]
    rec_papers = get_recommendations(
        positive_papers=top_seeds,
        negative_papers=negative_seeds,
        limit=50,
        research_question=state.topic,
    )

    # ── Step 6: merge + deduplicate + cap ────────────────────────────────────
    for p in rec_papers:
        if p["id"] not in seen_ids:
            seen_ids.add(p["id"])
            papers.append(p)
    papers.sort(key=lambda p: p.get("relevance_score", 0), reverse=True)
    papers = papers[:MAX_RESULTS]
    print(f"📚 Final corpus: {len(papers)} papers")

    # ── Step 7: parallel PDF downloads ──────────────────────────────────────
    downloaded = _download_papers_parallel(papers, DOWNLOAD_DIR)

    # ── Step 8: chunk → embed → KG ───────────────────────────────────────────
    for paper in papers:
        filepath = downloaded.get(paper["id"])
        if not filepath:
            continue
        chunks = extract_and_chunk_pdf(filepath)
        store_chunks_in_db(chunks, paper["title"], paper["id"])
        method, gap = extract_paper_insights(paper["title"])
        print(f"   → Method: {method}")
        print(f"   → Gap:    {gap}")
        update_knowledge_graph(paper["title"], method, gap)

    visualize_graph()

    # ── Write to state (NO temp files) ───────────────────────────────────────
    state.literature = {
        "queries":          queries,
        "primary_query":    queries[0] if queries else state.topic,
        "papers":           papers,
        "downloaded_count": len(downloaded),
        "kg_nodes": {
            "papers":  [n for n, d in kg.nodes(data=True) if d.get("type") == "Paper"],
            "methods": [n for n, d in kg.nodes(data=True) if d.get("type") == "Method"],
            "gaps":    [n for n, d in kg.nodes(data=True) if d.get("type") == "Gap"],
        },
        "chroma_count": collection.count(),
    }

    print("\n💾 Literature data stored in state.literature (no temp files).")
    print("🎉 Literature Review complete!")
    return state

## Agent 2: Hypothesis Generation Agent

In [ ]:
MAX_CONTEXT_CHUNKS     = 8
MAX_HYPOTHESIS_RETRIES = 3


def _extract_kg_context():
    """Summarise KG gaps/methods into brief text for the hypothesis prompt."""
    try:
        if not kg or kg.number_of_nodes() == 0:
            return ""
        gaps    = [n for n, d in kg.nodes(data=True) if d.get("type") == "Gap"]
        methods = [n for n, d in kg.nodes(data=True) if d.get("type") == "Method"]
        lines   = []
        if gaps:    lines.append(f"Identified research gaps: {'; '.join(gaps[:5])}")
        if methods: lines.append(f"Observed methods: {'; '.join(methods[:5])}")
        return "\n".join(lines)
    except Exception:
        return ""


def retrieve_research_context(topic):
    """Multi-angle vector retrieval — four query angles surface broader context."""
    print(f"🔎 Scanning vector store for context on: '{topic}'...")
    query_angles = [
        f"limitations, future work, research gaps in {topic}",
        f"key methodology and experimental design for {topic}",
        f"state of the art results and benchmarks for {topic}",
        f"open problems and unanswered questions in {topic}",
    ]
    per_angle = max(1, MAX_CONTEXT_CHUNKS // len(query_angles))
    seen_sigs, all_docs = set(), []
    for angle in query_angles:
        try:
            results = collection.query(query_texts=[angle], n_results=per_angle + 1)
            for doc in (results["documents"] or [[]])[0]:
                sig = doc[:100]
                if sig not in seen_sigs:
                    seen_sigs.add(sig)
                    all_docs.append(doc)
        except Exception as e:
            print(f"   ⚠️ Query failed: {e}")
    kg_context = _extract_kg_context()
    if not all_docs and not kg_context:
        print("⚠️ No relevant data found.")
        return None
    context_parts = ["\n\n---\n\n".join(all_docs[:MAX_CONTEXT_CHUNKS])]
    if kg_context:
        context_parts.append(f"\n\n--- Knowledge Graph Insights ---\n{kg_context}")
    print(f"📖 Retrieved {len(all_docs)} chunks" +
          (f" + KG ({len(kg_context)} chars)" if kg_context else ""))
    return "\n".join(context_parts)


def _validate_hypothesis_format(text):
    required = ["### Hypothesis", "**Statement:**", "**Grounding:**",
                "**Proposed Method:**", "**Novelty Score:**", "**Feasibility Score:**"]
    missing  = [r for r in required if r not in text]
    return len(missing) == 0, missing


def _parse_hypotheses(text):
    """Parse LLM output into a list of typed hypothesis dicts.

    Handles format variants the model may emit:
      • ### Hypothesis [1]: Title   (canonical — brackets around number)
      • ### Hypothesis 1: Title     (number without brackets)
      • ### Hypothesis: Title       (no number at all)

    Field lookaheads are anchored to a line-starting **Capital header so
    inline bold words inside field values are captured in full rather than
    truncating at the first '**' encountered.

    Score patterns accept bare digit, [digit], or digit/10 variants.
    """
    hypotheses = []
    # Flexible split: accept [N]:, N:, or bare : after "Hypothesis"
    blocks = re.split(r"###\s*Hypothesis\s*(?:\[?\d+\]?\s*:|\s*:)\s*", text)
    for block in blocks[1:]:
        hyp     = {}
        title_m = re.match(r"\s*(.+?)[\n\r]", block)
        hyp["title"] = title_m.group(1).strip() if title_m else "Untitled"
        # Stop only at a field header (**CapitalLetter) or end-of-text so
        # inline bold words inside a value don't truncate the capture.
        for pattern, key in [
            (r"\*\*Statement:\*\*\s*(.+?)(?=\n\s*\*\*[A-Z]|\Z)",       "statement"),
            (r"\*\*Grounding:\*\*\s*(.+?)(?=\n\s*\*\*[A-Z]|\Z)",       "grounding"),
            (r"\*\*Proposed Method:\*\*\s*(.+?)(?=\n\s*\*\*[A-Z]|\Z)", "proposed_method"),
        ]:
            m = re.search(pattern, block, re.DOTALL)
            hyp[key] = m.group(1).strip() if m else ""
        # Accept score as bare digit, [digit], or digit/10
        for label, key in [("Novelty Score", "novelty"), ("Feasibility Score", "feasibility")]:
            m = re.search(rf"\*\*{label}:\*\*\s*\[?(\d+)\]?", block)
            hyp[key] = int(m.group(1)) if m else 5
        hyp["combined_score"] = round(hyp["novelty"] * 0.6 + hyp["feasibility"] * 0.4, 2)
        hyp["raw_block"]      = "### Hypothesis: " + block
        hypotheses.append(hyp)
    return hypotheses


def generate_and_rank_hypotheses(topic, context):
    """Generate + validate hypotheses with a retry/repair loop."""
    print("🧠 Synthesising context into testable propositions...")
    model, processor = llm_manager.get_model()
    system_prompt = (
        "You are an elite principal investigator. Analyse literature excerpts, identify "
        "unaddressed technical challenges, and propose exactly THREE novel empirical hypotheses."
    )
    user_prompt = f"""
Research Core: {topic}

Extracted Literature Context:
{context[:3000]}

Generate 3 clear hypotheses in EXACTLY this format for every hypothesis:

### Hypothesis [Number]: [Short Title]
**Statement:** [Clear, testable hypothesis statement]
**Grounding:** [Trace directly back to limitations in the text]
**Proposed Method:** [One sentence on empirical execution]
**Novelty Score:** [1-10]
**Feasibility Score:** [1-10]
"""
    messages   = [{"role":"system","content":system_prompt},
                  {"role":"user","content":user_prompt}]
    output_text = ""
    for attempt in range(1, MAX_HYPOTHESIS_RETRIES + 1):
        torch.cuda.empty_cache()
        inputs    = processor.apply_chat_template(
            messages, tokenize=True, return_dict=True,
            return_tensors="pt", add_generation_prompt=True
        ).to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=800, temperature=0.7)
        output_text = processor.decode(
            outputs[0][input_len:], skip_special_tokens=True
        ).strip()
        is_valid, missing = _validate_hypothesis_format(output_text)
        if is_valid:
            print(f"✅ Valid format on attempt {attempt}/{MAX_HYPOTHESIS_RETRIES}")
            break
        print(f"⚠️ Attempt {attempt}: missing {missing}. Retrying...")
        messages.append({"role":"assistant","content":output_text})
        messages.append({"role":"user","content":(
            f"Your response was missing: {missing}. "
            "Regenerate ALL 3 hypotheses with the EXACT format above."
        )})
    else:
        print(f"⚠️ Format validation did not pass. Using last output.")
    return output_text

In [ ]:
def run_hypothesis_agent(state: ResearchState) -> ResearchState:
    """
    Agent 2 — Hypothesis Generation.

    Reads  : state.topic, state.literature (KG context via global kg / collection)
    Writes : state.hypothesis
    """
    print("\n=== Agent 2: Hypothesis Generation Agent ===")

    if collection.count() == 0:
        print("❌ Vector store is empty — run the Literature Agent first.")
        return state

    try:
        topic   = state.topic or state.literature.get("primary_query", "")
        context = retrieve_research_context(topic)
        if not context:
            return state

        torch.cuda.empty_cache()
        raw_output = generate_and_rank_hypotheses(topic, context)
        hypotheses = _parse_hypotheses(raw_output)

        if hypotheses:
            ranked          = sorted(enumerate(hypotheses),
                                     key=lambda x: x[1]["combined_score"], reverse=True)
            best_idx, best_hyp = ranked[0]

            print("\n" + "="*50)
            print("🌟 PROPOSED HYPOTHESES (ranked by score)")
            print("="*50)
            for rank, (i, h) in enumerate(ranked, 1):
                marker = "  ← SELECTED" if i == best_idx else ""
                print(f"\n  #{rank}  Score {h['combined_score']:.1f}  |  "
                      f"Novelty {h['novelty']}/10  |  Feasibility {h['feasibility']}/10")
                print(f"  Title: {h['title']}{marker}")
                print(f"  Statement: {h['statement'][:120]}...")

            log_entry = {
                "topic": topic,
                "num_hypotheses": len(hypotheses),
                "hypotheses": [{"title": h["title"], "novelty": h["novelty"],
                                "feasibility": h["feasibility"],
                                "combined_score": h["combined_score"]}
                               for h in hypotheses],
                "selected_index": best_idx,
                "selected_title": best_hyp["title"],
            }
        else:
            print("⚠️ Could not parse structured hypotheses — using raw output as fallback.")
            best_hyp  = {"title": "Unparsed", "statement": raw_output,
                         "grounding": "", "proposed_method": "",
                         "novelty": 5, "feasibility": 5, "raw_block": raw_output}
            best_idx  = 0
            log_entry = {"topic": topic, "num_hypotheses": 0}

        # ── Write to state (NO temp files) ───────────────────────────────────
        state.hypothesis = {
            "raw_output":  raw_output,
            "hypotheses":  hypotheses,
            "best":        best_hyp,
            "best_idx":    best_idx,
            "log":         log_entry,
        }

        print(f"\n💾 Hypothesis stored in state.hypothesis (no temp files).")

    except Exception as e:
        print(f"\n❌ Hypothesis Agent failed: {e}")
        _traceback.print_exc()
    finally:
        llm_manager.unload()

    return state

## Agent 3: Experiment Agent

In [ ]:
MAX_RETRIES      = 3
TIMEOUT_SECONDS  = 600
MAX_MEMORY_BYTES = 12 * 1024 ** 3
MAX_CPU_SECONDS  = TIMEOUT_SECONDS + 30

DANGEROUS_PATTERNS = [
    r"\bos\.system\s*\(",  r"\bsubprocess\.",  r"\bshutil\.rmtree\s*\(",
    r"\beval\s*\(",          r"\bexec\s*\(",    r"\bos\.remove\s*\(",
    r"\bos\.unlink\s*\(",   r"__import__\s*\(", r"\bsocket\.",
    r"\bctypes\.",            r'os\.environ\[[^\]]*(KEY|TOKEN|SECRET)',
]


def clean_llm_code(response: str) -> str:
    response   = response.strip()
    code_match = re.search(r'\x60\x60\x60(?:python|py)?\s*\n(.*?)(?:\x60\x60\x60|$)',
                           response, re.DOTALL | re.IGNORECASE)
    if code_match:
        return code_match.group(1).strip()
    lines = response.splitlines()
    return "\n".join(l for l in lines if not l.strip().startswith("`" * 3)).strip()


def validate_python_syntax(code: str):
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"SyntaxError: {e.msg} (line {e.lineno})"


def scan_for_dangerous_code(code: str):
    return [p for p in DANGEROUS_PATTERNS if re.search(p, code)]


def generate_hf_search_query(hypothesis, model, processor):
    messages = [
        {"role":"system","content":"Turn a scientific hypothesis into a short HF dataset search query."},
        {"role":"user","content":f"Hypothesis:\n'{hypothesis}'\nOutput ONLY 1-3 words."},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20, temperature=0.1)
    query = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip().replace('"','')
    print(f"🎯 HF search query: '{query}'")
    return query


def search_huggingface_datasets(search_query):
    print(f"🔎 Scanning Hugging Face for '{search_query}'...")
    try:
        api      = HfApi()
        datasets = api.list_datasets(search=search_query, sort="downloads", limit=3)
        ids      = [d.id for d in datasets]
        if not ids:
            return ["glue", "imdb"]
        print(f"🤗 Discovered: {ids}")
        return ids
    except Exception as e:
        print(f"❌ HF search failed: {e}. Using fallbacks.")
        return ["glue", "imdb"]


def verify_dataset_exists(dataset_id):
    try:
        HfApi().dataset_info(dataset_id)
        return True
    except Exception:
        return False


def design_experiment_blueprint(hypothesis, discovered_datasets, model, processor):
    print("📐 Architecting experiment blueprint...")
    verified = [d for d in discovered_datasets if verify_dataset_exists(d)]
    if not verified:
        verified = ["glue", "imdb"]
    user_prompt = f"""
Target Hypothesis:
"{hypothesis}"

Verified Hugging Face Datasets: {verified}

Format EXACTLY:
DATASETS: [best verified dataset]
BASELINES: [1-2 baseline models]
METRICS: [evaluation metrics]
COMPUTE: [batch size, precision, GPU constraints]
METHODOLOGY: [data loading and scoring outline]
"""
    messages  = [
        {"role":"system","content":"You are an ML Scientist. Design a concrete experiment blueprint."},
        {"role":"user","content":user_prompt},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.2)
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def generate_runnable_code(blueprint, model, processor):
    print("💻 Generating executable Python code...")
    user_prompt = f"""
Blueprint:
{blueprint}

Write a complete, standalone Python script. Requirements:
1. Use datasets.load_dataset() for the specified dataset.
2. Use valid sklearn metrics (accuracy_score, f1_score, mean_squared_error).
3. Import sys, traceback, json, os, torch explicitly.
4. device = 'cuda' if torch.cuda.is_available() else 'cpu'
5. Set random seed 42 for random, numpy, and torch.
6. Last printed line: RESULTS_JSON: {{"metric_name": value, ...}}
7. Wrap in a single ```python block.
"""
    messages  = [
        {"role":"system","content":"You are a Senior ML Engineer. Write clean, runnable Python."},
        {"role":"user","content":user_prompt},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=2500, temperature=0.1)
    return clean_llm_code(
        processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    )


def analyze_and_tune_script(script_path, error_message):
    """Self-repair: feed the error back to Gemma and get a corrected script."""
    with open(script_path, "r", encoding="utf-8") as f:
        current_code = f.read()
    model, processor = llm_manager.get_model()
    user_prompt = f"""
Broken script:
```python
{current_code}
```
Error:
{error_message}

INSTRUCTIONS:
1. If CUDA OOM: halve batch_size or add gradient accumulation.
2. Never invent optimizer params.
3. Keep fixed seed and RESULTS_JSON stdout line.
4. Output ONLY corrected code in a single ```python block.
"""
    messages  = [
        {"role":"system","content":"You are an MLOps debugging engineer. Fix broken ML code."},
        {"role":"user","content":user_prompt},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    print("🤖 Repairing code...")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=2500, temperature=0.2)
    corrected = clean_llm_code(
        processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    )
    is_valid, syntax_err = validate_python_syntax(corrected)
    if not is_valid:
        print(f"⚠️ Repair produced invalid Python ({syntax_err}). Keeping previous version.")
        llm_manager.unload()
        return False
    if scan_for_dangerous_code(corrected):
        print("🛑 Repair introduced disallowed patterns. Keeping previous version.")
        llm_manager.unload()
        return False
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(corrected)
    print(f"💾 Repaired script written.")
    llm_manager.unload()
    return True


def extract_results_json(stdout_text):
    for line in reversed(stdout_text.splitlines()):
        if line.strip().startswith("RESULTS_JSON:"):
            payload = line.split("RESULTS_JSON:", 1)[1].strip()
            try:
                return json.loads(payload)
            except json.JSONDecodeError:
                return None
    return None


def _build_sandbox_env():
    passthrough = ("PATH","HOME","LD_LIBRARY_PATH","CUDA_VISIBLE_DEVICES",
                   "PYTORCH_CUDA_ALLOC_CONF","PYTORCH_ALLOC_CONF")
    return {k: os.environ[k] for k in passthrough if k in os.environ}


def _resource_limit_preexec():
    try:
        import resource
        resource.setrlimit(resource.RLIMIT_CPU, (MAX_CPU_SECONDS, MAX_CPU_SECONDS))
        resource.setrlimit(resource.RLIMIT_AS,  (MAX_MEMORY_BYTES, MAX_MEMORY_BYTES))
    except Exception:
        pass

In [ ]:
def run_experiment_agent(state: ResearchState) -> ResearchState:
    """
    Agent 3 — Experiment Design & Execution.

    Reads  : state.hypothesis
    Writes : state.experiment
    Note   : Still writes a TEMPORARY script to disk for subprocess execution.
             This is infrastructure, not an inter-agent handoff.
    """
    print("\n=== Agent 3: Experiment Agent ===")

    # ── Reconstruct hypothesis text from state (replaces file read) ──────────
    best            = state.hypothesis.get("best", {}) if state.hypothesis else {}
    hypothesis_text = (
        f"### Selected Hypothesis: {best.get('title','')}\n\n"
        f"**Statement:** {best.get('statement','')}\n\n"
        f"**Grounding:** {best.get('grounding','')}\n\n"
        f"**Proposed Method:** {best.get('proposed_method','')}\n\n"
        f"**Novelty Score:** {best.get('novelty','')}\n"
        f"**Feasibility Score:** {best.get('feasibility','')}\n"
    ) if best else input("Paste the hypothesis to test:\n> ")

    TARGET_SCRIPT = os.path.join(SANDBOX_DIR, "generated_experiment.py")
    os.makedirs(SANDBOX_DIR, exist_ok=True)

    # ── Generate code ─────────────────────────────────────────────────────────
    code_str = None
    try:
        model, processor = llm_manager.get_model()
        hf_query     = generate_hf_search_query(hypothesis_text, model, processor)
        hf_datasets  = search_huggingface_datasets(hf_query)
        torch.cuda.empty_cache()
        blueprint    = design_experiment_blueprint(hypothesis_text, hf_datasets, model, processor)
        torch.cuda.empty_cache()
        code_str     = generate_runnable_code(blueprint, model, processor)

        is_valid, syntax_err = validate_python_syntax(code_str)
        if not is_valid:
            print(f"❌ Generated code failed static validation ({syntax_err}).")
            state.experiment = {"final_status": "GENERATION_FAILED", "code": code_str, "runs": []}
            return state
        if scan_for_dangerous_code(code_str):
            print("🛑 Generated code contains disallowed patterns. Refusing to execute.")
            state.experiment = {"final_status": "REFUSED", "code": code_str, "runs": []}
            return state

        with open(TARGET_SCRIPT, "w", encoding="utf-8") as f:
            f.write(code_str)
        print(f"✅ Experiment script written to {TARGET_SCRIPT}")
    except Exception as e:
        print(f"❌ Generation phase failed: {e}")
        state.experiment = {"final_status": "GENERATION_FAILED", "code": "", "runs": []}
        return state
    finally:
        llm_manager.unload()

    # ── Sandbox execution loop ────────────────────────────────────────────────
    sandbox_env  = _build_sandbox_env()
    run_log      = []
    final_status = "FAILED"
    final_metrics = None

    for run_attempt in range(1, MAX_RETRIES + 1):
        print(f"\n🚀 [Attempt {run_attempt}/{MAX_RETRIES}] Sandboxed run...")
        start_time = time.monotonic()
        try:
            process = subprocess.run(
                [sys.executable, os.path.basename(TARGET_SCRIPT)],
                capture_output=True, text=True, timeout=TIMEOUT_SECONDS,
                cwd=SANDBOX_DIR, env=sandbox_env,
                preexec_fn=_resource_limit_preexec if os.name == "posix" else None,
            )
            duration = time.monotonic() - start_time
            entry    = {
                "run_id": run_attempt, "duration_seconds": round(duration, 2),
                "status": "SUCCESS" if process.returncode == 0 else "FAILED",
                "stdout_summary": process.stdout[-800:].strip(),
                "stderr_summary": process.stderr[-800:].strip(),
                "metrics": extract_results_json(process.stdout or ""),
            }
            run_log.append(entry)
            if process.returncode == 0:
                print(f"🎉 SUCCESS on attempt {run_attempt}.")
                print(f"📈 Output:\n{process.stdout[-400:]}")
                final_status  = "SUCCESS"
                final_metrics = entry["metrics"]
                break
            else:
                print(f"⚠️ Exit code {process.returncode}. Stderr:\n{process.stderr[-300:]}")
                if run_attempt < MAX_RETRIES:
                    try:
                        repaired = analyze_and_tune_script(TARGET_SCRIPT, process.stderr)
                        if repaired:
                            print("🔧 Script repaired — retrying.")
                    except Exception as re:
                        print(f"⚠️ Repair raised exception: {re}")

        except subprocess.TimeoutExpired as te:
            duration  = time.monotonic() - start_time
            te_stdout = te.stdout.decode() if isinstance(te.stdout, bytes) else (te.stdout or "")
            entry     = {
                "run_id": run_attempt, "duration_seconds": round(duration, 2),
                "status": "TIMEOUT", "stdout_summary": te_stdout[-800:],
                "stderr_summary": f"Timeout after {TIMEOUT_SECONDS}s", "metrics": None,
            }
            run_log.append(entry)
            print(f"❌ Timeout on attempt {run_attempt}.")
            if run_attempt < MAX_RETRIES:
                try:
                    analyze_and_tune_script(TARGET_SCRIPT,
                                            f"RuntimeError: Timeout after {TIMEOUT_SECONDS}s")
                except Exception as re:
                    print(f"⚠️ Repair raised exception: {re}")
        except Exception as e:
            print(f"❌ Sandbox engine failure: {e}")
            break

    # ── Write to state (NO results_log.json handoff) ─────────────────────────
    state.experiment = {
        "code":          code_str,
        "runs":          run_log,
        "final_status":  final_status,
        "metrics":       final_metrics,
        "script_path":   TARGET_SCRIPT,
    }

    print(f"\n💾 Experiment results stored in state.experiment (no temp files).")
    return state

## Agent 4: Analysis Agent

In [ ]:
def _parse_analysis_field(text, field):
    for line in text.split("\n"):
        if line.strip().startswith(f"{field}:"):
            return line.replace(f"{field}:", "").strip()
    return ""


def analyze_results_vs_hypothesis(hypothesis_text, runs, topic):
    """Uses Gemma to interpret experiment results against the hypothesis."""
    print("🧠 Interpreting results vs hypothesis...")
    model, processor = llm_manager.get_model()
    latest           = runs[-1] if runs else {}
    status           = latest.get("status", "UNKNOWN")
    stdout_summary   = latest.get("stdout_summary", "No output.")
    stderr_summary   = latest.get("stderr_summary", "No errors.")
    all_attempts     = len(runs)
    user_prompt = f"""
Hypothesis / Experiment:
{hypothesis_text[:2000]}

Execution Summary:
- Total Attempts: {all_attempts}
- Final Status  : {status}
- Output        : {stdout_summary}
- Errors        : {stderr_summary}

Provide analysis in EXACTLY this format:
VERDICT: [SUCCESS | FAILURE | PIVOT]
INTERPRETATION: [2-4 sentences on scientific meaning]
EVIDENCE: [Specific outputs or metrics cited]
LIMITATIONS: [Key limitations or confounds]
NEXT_STEP: [If PIVOT: modified hypothesis. If SUCCESS: confirm write-up. If FAILURE: new direction.]
CONFIDENCE: [Low | Medium | High]
"""
    messages  = [
        {"role":"system","content":
         "You are a senior research scientist. Analyse experiment results vs hypothesis."},
        {"role":"user","content":user_prompt},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=600, temperature=0.3)
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

In [ ]:
def run_analysis_agent(state: ResearchState) -> ResearchState:
    """
    Agent 4 — Results Analysis.

    Reads  : state.hypothesis, state.experiment, state.topic
    Writes : state.analysis
             If PIVOT: also updates state.hypothesis['best'] with revised hypothesis.
    """
    print("\n=== Agent 4: Analysis Agent ===")

    if not state.experiment:
        print("❌ No experiment results — run the Experiment Agent first.")
        return state

    # Reconstruct hypothesis text from state (replaces file read)
    best = state.hypothesis.get("best", {}) if state.hypothesis else {}
    hypothesis_text = (
        f"### Hypothesis: {best.get('title','')}\n"
        f"**Statement:** {best.get('statement','')}\n"
        f"**Proposed Method:** {best.get('proposed_method','')}\n"
    ) if best else "No hypothesis available."

    runs = state.experiment.get("runs", [])

    try:
        analysis_text = analyze_results_vs_hypothesis(hypothesis_text, runs, state.topic)

        print("\n" + "="*50)
        print("📊 ANALYSIS REPORT")
        print("="*50 + "\n")
        print(analysis_text)

        verdict    = _parse_analysis_field(analysis_text, "VERDICT").upper()
        next_step  = _parse_analysis_field(analysis_text, "NEXT_STEP")
        confidence = _parse_analysis_field(analysis_text, "CONFIDENCE")

        # ── Write to state (NO analysis_report.txt handoff) ──────────────────
        state.analysis = {
            "report":     analysis_text,
            "verdict":    verdict,
            "next_step":  next_step,
            "confidence": confidence,
        }

        print(f"\n🏁 Verdict: {verdict}  |  Confidence: {confidence}")

        if verdict == "SUCCESS":
            print("✅ Ready for Writing Agent.")
        elif verdict == "FAILURE":
            print("❌ Hypothesis refuted — consider a new research direction.")
        elif verdict == "PIVOT":
            print(f"\n🔄 PIVOT: {next_step}")
            # Update hypothesis best with the revised direction (no file write)
            if next_step and state.hypothesis:
                state.hypothesis["best"] = {
                    **state.hypothesis.get("best", {}),
                    "title":     "Revised (Auto-generated by Analysis Agent)",
                    "statement": next_step,
                }
                print("💾 Revised hypothesis stored in state.hypothesis['best'].")
        else:
            print("⚠️ Could not determine a clear verdict.")

        print("\n💾 Analysis stored in state.analysis (no temp files).")

    except Exception as e:
        print(f"\n❌ Analysis Agent failed: {e}")
        _traceback.print_exc()
    finally:
        llm_manager.unload()

    return state

## Agent 5: Writing Agent

In [ ]:
def compile_latex_paper(sections, cited_works, topic):
    def escape(s):
        for ch, rep in [("&",r"\&"),("%",r"\%"),("#",r"\#")]:
            s = s.replace(ch, rep)
        return s
    bib = "\n".join(
        f"\\bibitem{{paper{i+1}}} {escape(t)}."
        for i, t in enumerate(cited_works)
    )
    body = ""
    for sec, latex_cmd in [
        ("Abstract",     "abstract"),
        ("Introduction", "section{Introduction}"),
        ("Related Work", "section{Related Work}"),
        ("Methodology",  "section{Methodology}"),
        ("Results",      "section{Results}"),
        ("Discussion",   "section{Discussion}"),
        ("Conclusion",   "section{Conclusion}"),
    ]:
        content = sections.get(sec, "Not generated.")
        if sec == "Abstract":
            body += f"\n\\begin{{abstract}}\n{content}\n\\end{{abstract}}\n"
        else:
            body += f"\n\\{latex_cmd}\n{content}\n"
    return (
        f"\\documentclass[11pt,a4paper]{{article}}\n"
        f"\\usepackage{{amsmath,amssymb,graphicx,booktabs}}\n"
        f"\\usepackage[colorlinks=true,linkcolor=blue,citecolor=blue,urlcolor=blue]{{hyperref}}\n"
        f"\\usepackage[numbers,sort&compress]{{natbib}}\n"
        f"\\usepackage{{microtype}}\n"
        f"\\usepackage[T1]{{fontenc}}\n"
        f"\\usepackage[margin=1in]{{geometry}}\n"
        f"\\title{{{escape(topic.replace('_',' ').title())}}}\n"
        f"\\author{{AI Research Pipeline}}\n"
        f"\\date{{\\today}}\n\n"
        f"\\begin{{document}}\n\\maketitle\n"
        f"{body}\n"
        f"\\begin{{thebibliography}}{{99}}\n{bib}\n\\end{{thebibliography}}\n"
        f"\\end{{document}}\n"
    )


In [ ]:
def run_writing_agent(state: ResearchState) -> ResearchState:
    """
    Agent 5 — Paper Writing.

    Reads  : state.topic, state.hypothesis, state.experiment, state.analysis, state.literature
    Writes : state.paper  (LaTeX string — no intermediate files)
    """
    print("\n=== Agent 5: Writing Agent ===")

    best            = state.hypothesis.get("best", {}) if state.hypothesis else {}
    hypothesis_text = (f"**Statement:** {best.get('statement','')}\n"
                       f"**Method:** {best.get('proposed_method','')}") if best else "No hypothesis."

    analysis_report  = state.analysis.get("report",  "No analysis.") if state.analysis else "No analysis."
    latest_run       = state.experiment.get("runs", [{}])[-1] if state.experiment else {}
    results_summary  = latest_run.get("stdout_summary", "No results available.")
    research_topic   = state.topic or state.literature.get("primary_query", "Research")

    cited_works = retrieve_literature_for_citation(research_topic)
    lit_context = "\n".join(f"- {t}: {e[:150]}..." for t, e in cited_works.items())

    model, processor = llm_manager.get_model()
    torch.cuda.empty_cache()

    base_ctx = (
        f"Research Topic: {research_topic}\n"
        f"Hypothesis: {hypothesis_text[:500]}\n"
        f"Key Literature:\n{lit_context}\n"
        f"Analysis:\n{analysis_report[:800]}\n"
        f"Experiment Output:\n{results_summary[:500]}"
    )

    section_contexts = {
        "Abstract":     base_ctx,
        "Introduction": f"Research topic and motivation:\n{base_ctx}",
        "Related Work": f"Cite and contrast these papers:\n{lit_context}\n\nTopic: {research_topic}",
        "Methodology":  f"Describe the experimental methodology:\n{hypothesis_text[:800]}\nTopic: {research_topic}",
        "Results":      f"Present the results:\n{results_summary}\nAnalysis:\n{analysis_report[:600]}",
        "Discussion":   f"Interpret findings:\n{analysis_report}\nResults: {results_summary[:400]}",
        "Conclusion":   base_ctx,
    }
    section_order = ["Abstract","Introduction","Related Work",
                     "Methodology","Results","Discussion","Conclusion"]

    sections = {}
    for section in section_order:
        torch.cuda.empty_cache()
        draft   = draft_section(section, section_contexts[section], model, processor)
        refined = refine_section(section, draft, model, processor)
        sections[section] = refined
        print(f"   ✅ {section} complete ({len(refined)} chars)")

    # ── Write to state (NO paper_draft.txt / research_paper.tex handoff) ─────
    state.paper = compile_latex_paper(sections, cited_works, research_topic)

    print(f"\n💾 Paper stored in state.paper ({len(state.paper)} chars, no temp files).")
    print("🎉 Writing Agent complete!")
    llm_manager.unload()
    return state

## Agent 6: Critic / Review Agent

In [ ]:
def apply_reviewer_revisions(paper_latex, meta_review, model, processor):
    print("\n✏️  Applying reviewer feedback to produce a revised LaTeX draft...")
    user_prompt = (
        "You are revising a LaTeX research paper based on peer-review feedback.\n\n"
        f"Original LaTeX (preserve \\documentclass, \\usepackage lines, \\title, "
        f"\\author, \\date, and \\begin{{thebibliography}} block exactly):\n"
        f"{paper_latex[:2500]}\n\n"
        f"Reviewer Feedback:\n{meta_review[:1500]}\n\n"
        "Instructions:\n"
        "1. Address every MAJOR_ISSUES point.\n"
        "2. Incorporate MINOR_ISSUES where feasible.\n"
        "3. Improve clarity per CLARITY feedback.\n"
        "4. Strengthen methodology per TECHNICAL_QUALITY feedback.\n"
        "5. Add missing reproducibility details to the Methodology section.\n"
        "6. Add \\textit{\\small Revised following peer review.} immediately after \\maketitle.\n"
        "\nOutput the COMPLETE revised paper as a valid, compilable LaTeX document.\n"
        "Start with \\documentclass and end with \\end{document}.\\n"
        "Do NOT include any explanation outside the LaTeX source.\n"
    )
    messages  = [
        {"role":"system","content":
         "You are an expert academic author. Output only valid LaTeX. No markdown, no prose outside the document."},
        {"role":"user","content":user_prompt},
    ]
    inputs    = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=3000, temperature=0.3)
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


In [ ]:
def _is_valid_latex(text: str) -> bool:
    """Heuristic check: does the string look like a complete LaTeX document?"""
    t = text.strip()
    return t.startswith(r"\documentclass") and r"\end{document}" in t

def save_paper_to_file(state, output_dir: str = ".") -> str:
    """
    Write state.paper (LaTeX source) to <topic>.tex in output_dir.
    Returns the absolute path of the saved file.
    """
    if not state.paper:
        print("⚠️  No paper in state to save.")
        return None

    safe_topic = re.sub(r'[^\w\s-]', '', state.topic or 'research_paper')
    safe_topic = re.sub(r'\s+', '_', safe_topic.strip())[:60] or 'research_paper'
    filename   = f"{safe_topic}.tex"
    filepath   = os.path.join(output_dir, filename)

    os.makedirs(output_dir, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as fh:
        fh.write(state.paper)

    abs_path = os.path.abspath(filepath)
    print(f"\n📄 LaTeX paper saved  : {abs_path}")
    print(f"   Compile with      : pdflatex \"{filename}\"")
    print(f"   (run twice for cross-references / TOC)")
    return abs_path

def run_critic_agent(state: ResearchState) -> ResearchState:
    """
    Agent 6 — Peer Review & Revision.

    Reads  : state.paper
    Writes : state.paper  (revised LaTeX — full document, valid for compilation)
             state.analysis['review']         (peer review text)
             state.analysis['recommendation'] (overall recommendation)

    Also saves state.paper to <topic>.tex via save_paper_to_file().
    """
    print("\n=== Agent 6: Critic / Review Agent ===")

    if not state.paper:
        print("❌ No paper found — run the Writing Agent first.")
        return state

    model, processor = llm_manager.get_model()
    sections         = _parse_plain_sections(state.paper)
    full_review      = "="*60 + "\nPEER REVIEW REPORT\n" + "="*60 + "\n"
    section_scores   = []

    # ── Per-section critiques ───────────────────────────────────────────────────────
    for sec in ["INTRODUCTION", "METHODOLOGY", "RESULTS", "DISCUSSION"]:
        content = sections.get(sec, "")
        if not content:
            continue

        torch.cuda.empty_cache()
        critique = critique_section(sec.title(), content, model, processor)
        full_review += f"\n{'\u2500'*40}\nSECTION: {sec}\n{'\u2500'*40}\n{critique}\n"

        for line in critique.split("\n"):
            if line.strip().startswith("SCORE:"):
                try:
                    section_scores.append(int(line.replace("SCORE:", "").strip()[0]))
                except Exception:
                    pass

    # ── Meta-review ─────────────────────────────────────────────────────────────────
    torch.cuda.empty_cache()
    meta_review  = run_meta_review(state.paper, model, processor)
    full_review += f"\n\n{'='*60}\nMETA-REVIEW\n{'='*60}\n{meta_review}\n"

    if section_scores:
        avg = sum(section_scores) / len(section_scores)
        full_review += f"\nAverage Section Score: {avg:.1f} / 4.0\n"

    recommendation = _parse_analysis_field(meta_review, "OVERALL_RECOMMENDATION")
    print(f"\n🏛️  Reviewer Recommendation: {recommendation}")

    # ── Revision pass ───────────────────────────────────────────────────────────────
    torch.cuda.empty_cache()
    revised_text = apply_reviewer_revisions(state.paper, meta_review, model, processor)

    # Use the full revised LaTeX if the model returned a valid document;
    # otherwise fallback to patching the original with a revision note.
    if _is_valid_latex(revised_text):
        print("✅ Revised LaTeX document validated.")
        revised_latex = revised_text
    else:
        print("⚠️  Revision did not return valid LaTeX — patching original with revision note.")
        revised_latex = state.paper.replace(
            r"\begin{document}",
            r"\begin{document}" + "\n\\textit{\\small Revised following peer review.}\n\\vspace{1em}\n"
        )

    # ── Write to state ──────────────────────────────────────────────────────────────
    state.paper = revised_latex
    if state.analysis is None:
        state.analysis = {}
        
    state.analysis['review']         = full_review
    state.analysis['recommendation'] = recommendation

    print("\n💾 Revised paper stored in state.paper.")
    print("💾 Review stored in state.analysis['review'].")

    print("\n✅ Critic / Review Agent complete!")
    llm_manager.unload()
    return state

## Full Pipeline

In [ ]:
def run_research_pipeline(topic: str = "") -> ResearchState:
    """
    End-to-end research pipeline.

    All inter-agent communication is carried exclusively through the
    ResearchState object — no temporary files cross agent boundaries.

    Args
    ----
    topic : Optional pre-set research question.
            If empty, Agent 1 will prompt the user interactively.

    Returns
    -------
    ResearchState with all fields populated.
    The final LaTeX paper is written to <topic>.tex in the working directory.
    """
    state = ResearchState(topic=topic)

    print("\n" + "="*60)
    print("  ResearchAgent v3 — State-Based Pipeline")
    print("="*60 + "\n")

    state = run_literature_agent(state)
    print(f"\n📌 State after Literature Agent:")
    state.summary()

    state = run_hypothesis_agent(state)
    print(f"\n📌 State after Hypothesis Agent:")
    state.summary()

    state = run_experiment_agent(state)
    print(f"\n📌 State after Experiment Agent:")
    state.summary()

    state = run_analysis_agent(state)
    print(f"\n📌 State after Analysis Agent:")
    state.summary()

    # Auto-pivot: re-run experiment + analysis with the revised hypothesis.
    # Guard against state.analysis being None if the agent errored out.
    if (state.analysis or {}).get("verdict") == "PIVOT":
        print("\n🔄 PIVOT detected — re-running experiment with revised hypothesis...")
        state = run_experiment_agent(state)
        print(f"\n📌 State after Experiment Agent (pivot):")
        state.summary()
        state = run_analysis_agent(state)
        print(f"\n📌 State after Analysis Agent (pivot):")
        state.summary()

    state = run_writing_agent(state)
    print(f"\n📌 State after Writing Agent:")
    state.summary()

    state = run_critic_agent(state)
    print(f"\n📌 State after Critic Agent:")
    state.summary()

    # ── Single save point: write the final LaTeX paper to disk ───────────
    if state.paper:
        save_paper_to_file(state)
    else:
        print("\n⚠️  Pipeline finished but state.paper is empty — no .tex file written.")

    print("\n" + "="*60)
    print("  🏁 Pipeline complete!")
    print("="*60)
    state.summary()

    return state


In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
# Edit the topic string, then run this cell to start the full pipeline.
final_state = run_research_pipeline(topic="How does the integration of adaptive step-size ODE solvers in the reverse sampling process affect the trade-off between inference speed and structural fidelity in high-resolution image synthesis?")
